Create data panel for Step 4

In [ ]:
# ============================================================
# Step 4 Model Panel Builder
# 目的：
# 构建一个完整的 Step 4 建模数据表 stage4_model_panel.parquet
#
# 每一行代表：
# 某一天 date、某只股票 instrument_id，在当天的可交易状态、
# 特征变量，以及下一晚 overnight return 作为预测目标。
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd


# ============================================================
# 0. 路径和基础参数设置
# 这一步是在定义输入文件在哪里、输出文件保存到哪里，
# 以及开发窗口只能到 2024-12-31。
# ============================================================

DATA_DIR = Path("data")

STAGE2_DIR = Path("stage123data")

STAGE3_DIR = Path("stage3_outputs")

OUT_DIR = Path("stage4_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = "2010-01-01"
END_DATE = "2024-12-31"

# 如果你担心 short interest 当天可用性不够保守，可以改成 True。
# 正常情况下 Stage 1 已经处理了 publication lag + vendor lag，所以这里设 False。
EXTRA_SHIFT_SHORT_INTEREST = False

# 可选：是否把 Stage 3 的 borrow tier 加进 Step 4 特征表
# 第一版模型可以设 False；如果想做 ablation，可以设 True。
ADD_STAGE3_BORROW_FEATURES = True


# ============================================================
# 1. 读取 Stage 2 eligibility table
# 这一步是在读取 Step 2 的主输出。
# 它定义了每天哪些股票可以交易，以及不能交易的原因。
# Step 4 的主表应该以这个文件为基础。
# ============================================================

stage2_path = STAGE2_DIR / "stage2_capacity_eligibility.parquet"

panel = pd.read_parquet(stage2_path)
panel["date"] = pd.to_datetime(panel["date"])
panel["instrument_id"] = panel["instrument_id"].astype("int64")

# 为了后续计算方便，先排序
panel = panel.sort_values(["instrument_id", "date"]).reset_index(drop=True)

print("Stage 2 panel loaded:")
print(panel.shape)
print(panel[["date", "instrument_id", "ticker", "is_trade_eligible", "eligibility_reason"]].head())


# ============================================================
# 2. 读取价格数据并计算 adjusted open / close 和收益率
# 这一步是在重新构造 Step 4 的预测目标。
#
# r_on  = 从昨天收盘到今天开盘的 overnight return
# r_id  = 从今天开盘到今天收盘的 intraday return
# r_cc  = 从昨天收盘到今天收盘的 close-to-close return
#
# Step 4 要预测的是：
# target_r_on_next = 下一天的 r_on
# 也就是 day t 收盘后到 day t+1 开盘的 overnight return。
# ============================================================

prices_path = DATA_DIR / "prices.parquet"
prices = pd.read_parquet(prices_path)

prices["date"] = pd.to_datetime(prices["date"])
prices["instrument_id"] = prices["instrument_id"].astype("int64")

# 去重并排序，防止同一只股票同一天有重复记录
prices = (
    prices
    .drop_duplicates(subset=["instrument_id", "date"])
    .sort_values(["instrument_id", "date"])
    .reset_index(drop=True)
)

# 计算调整因子
# adjusted_close = close * adj_factor
prices["adj_factor"] = prices["adjusted_close"] / prices["close"]

# 调整开盘价和收盘价，使它们和 adjusted_close 在同一口径下
prices["open_adj"] = prices["open"] * prices["adj_factor"]
prices["close_adj"] = prices["adjusted_close"]

# 计算上一交易日 adjusted close
prices["close_adj_lag1"] = prices.groupby("instrument_id")["close_adj"].shift(1)

# 三个收益率
prices["r_on"] = prices["open_adj"] / prices["close_adj_lag1"] - 1
prices["r_id"] = prices["close_adj"] / prices["open_adj"] - 1
prices["r_cc"] = prices["close_adj"] / prices["close_adj_lag1"] - 1

# Step 4 的预测目标：下一天 overnight return
prices["target_r_on_next"] = prices.groupby("instrument_id")["r_on"].shift(-1)

# 只保留后面需要用的价格和收益率列
price_cols = [
    "date", "instrument_id",
    "open_adj", "close_adj",
    "r_on", "r_id", "r_cc",
    "target_r_on_next"
]

prices_for_merge = prices[price_cols].copy()

print("\nPrice / return data prepared:")
print(prices_for_merge.shape)
print(prices_for_merge.head())


# ============================================================
# 3. 构造基础收益率特征
# 这一步是在构造 Step 4 的 baseline alpha features。
#
# 注意：
# - target 用 shift(-1)，因为它是未来真实结果。
# - feature 不能偷看未来。
# - r_id 和 r_cc 用到了当天 close，所以作为 feature 时必须 shift(1)。
# - r_on_t 在 day t 开盘后已经知道，理论上 15:50 可以使用。
# ============================================================

prices_feat = prices.copy()
g = prices_feat.groupby("instrument_id", group_keys=False)

# 当天 overnight return：day t 开盘后已经知道，所以 15:50 可以使用
prices_feat["r_on_today"] = prices_feat["r_on"]

# 昨天的 overnight / intraday / close-to-close return
prices_feat["r_on_lag1"] = g["r_on"].shift(1)
prices_feat["r_id_lag1"] = g["r_id"].shift(1)
prices_feat["r_cc_lag1"] = g["r_cc"].shift(1)

# 过去 overnight return 均值
# r_on 是开盘时已经知道的信息，这里 rolling 包含今天 r_on，仍然是可观察的。
prices_feat["r_on_mean_5"] = g["r_on"].transform(
    lambda s: s.rolling(5, min_periods=5).mean()
)
prices_feat["r_on_mean_20"] = g["r_on"].transform(
    lambda s: s.rolling(20, min_periods=20).mean()
)

# 过去 intraday return 均值
# r_id 用到了 close，所以必须 shift(1)，只能用到昨天及以前。
prices_feat["r_id_mean_5_lag1"] = g["r_id"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=5).mean()
)
prices_feat["r_id_mean_20_lag1"] = g["r_id"].transform(
    lambda s: s.shift(1).rolling(20, min_periods=20).mean()
)

# 过去 close-to-close return 均值 / momentum
# r_cc 用到了 close，所以必须 shift(1)。
prices_feat["r_cc_mean_5_lag1"] = g["r_cc"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=5).mean()
)
prices_feat["r_cc_mean_20_lag1"] = g["r_cc"].transform(
    lambda s: s.shift(1).rolling(20, min_periods=20).mean()
)

# 过去收益率波动率
# 同样只能用到昨天及以前的 r_cc。
prices_feat["vol_20_lag1"] = g["r_cc"].transform(
    lambda s: s.shift(1).rolling(20, min_periods=20).std()
)
prices_feat["vol_60_lag1"] = g["r_cc"].transform(
    lambda s: s.shift(1).rolling(60, min_periods=40).std()
)

# 极端 overnight move 特征
prices_feat["abs_r_on_today"] = prices_feat["r_on_today"].abs()
prices_feat["abs_r_cc_lag1"] = prices_feat["r_cc_lag1"].abs()

feature_price_cols = [
    "date", "instrument_id",
    "r_on_today",
    "r_on_lag1",
    "r_id_lag1",
    "r_cc_lag1",
    "r_on_mean_5",
    "r_on_mean_20",
    "r_id_mean_5_lag1",
    "r_id_mean_20_lag1",
    "r_cc_mean_5_lag1",
    "r_cc_mean_20_lag1",
    "vol_20_lag1",
    "vol_60_lag1",
    "abs_r_on_today",
    "abs_r_cc_lag1",
]

price_features_for_merge = prices_feat[feature_price_cols].copy()

print("\nPrice-based features prepared:")
print(price_features_for_merge.shape)
print(price_features_for_merge.head())


# ============================================================
# 4. 把价格、收益率、target 和基础特征合并到 Stage 2 主表
# 这一步是在形成 Step 4 数据文件的主体。
# 主表仍然是 Stage 2 eligibility table。
# ============================================================

panel = panel.merge(
    prices_for_merge,
    on=["date", "instrument_id"],
    how="left"
)

panel = panel.merge(
    price_features_for_merge,
    on=["date", "instrument_id"],
    how="left"
)

print("\nPanel after merging price returns and features:")
print(panel.shape)


# ============================================================
# 5. 加入 short interest 特征
# 这一步是在加入 Stage 1 处理好的日频 short interest 数据。
#
# dsi   = short interest ratio
# dtcn  = days to cover
# ddtcn = days-to-cover change
#
# 这些变量可以作为 Step 4 的 short-interest feature group。
# ============================================================

si_path = STAGE2_DIR / "stage1_short_interest_daily_from_stage1_2.parquet"

short_interest = pd.read_parquet(si_path)
short_interest = short_interest.rename(columns={"stock_id": "instrument_id"})

short_interest["date"] = pd.to_datetime(short_interest["date"])
short_interest["instrument_id"] = short_interest["instrument_id"].astype("int64")

short_interest = short_interest.sort_values(["instrument_id", "date"]).reset_index(drop=True)

# 可选：如果想更保守，可以额外 shift 一天，确保只用到前一天已知的 short interest。
if EXTRA_SHIFT_SHORT_INTEREST:
    si_g = short_interest.groupby("instrument_id", group_keys=False)
    short_interest[["dsi", "dtcn", "ddtcn"]] = si_g[["dsi", "dtcn", "ddtcn"]].shift(1)

panel = panel.merge(
    short_interest[["date", "instrument_id", "dsi", "dtcn", "ddtcn"]],
    on=["date", "instrument_id"],
    how="left"
)

print("\nPanel after merging short-interest features:")
print(panel.shape)


# ============================================================
# 6. 可选：加入 Stage 3 borrow tier 特征
# 这一步不是 Step 4 必须的。
#
# Stage 3 的主要用途是 Step 5 给 short leg 扣 borrow cost。
# 但你也可以把 borrow_tier / hard_to_borrow_flag 作为 Step 4 的辅助特征。
# ============================================================

borrow_path = STAGE3_DIR / "stage3_borrow_tiers.parquet"

if ADD_STAGE3_BORROW_FEATURES and borrow_path.exists():
    borrow = pd.read_parquet(borrow_path)
    borrow["date"] = pd.to_datetime(borrow["date"])
    borrow["instrument_id"] = borrow["instrument_id"].astype("int64")

    borrow_cols = [
        "date", "instrument_id",
        "hard_to_borrow_flag",
        "borrow_stress_flag",
        "no_short_interest",
        "borrow_tier",
        "borrow_rate_annual_bps",
        "borrow_cost_daily_bps",
    ]

    panel = panel.merge(
        borrow[borrow_cols],
        on=["date", "instrument_id"],
        how="left"
    )

    print("\nStage 3 borrow features merged.")
else:
    print("\nStage 3 borrow features not merged.")


# ============================================================
# 7. 加入日期类特征
# 这一步是在加入 calendar features。
# 日期类特征没有未来信息问题，可以直接从 date 构造。
# ============================================================

panel["month"] = panel["date"].dt.month
panel["day_of_week"] = panel["date"].dt.dayofweek
panel["is_month_end"] = panel["date"].dt.is_month_end.astype(int)
panel["is_quarter_end"] = panel["date"].dt.is_quarter_end.astype(int)


# ============================================================
# 8. 加入样本切分 sample_split
# 这一步是在给每一行打上 train / valid / test 标签。
#
# 你后面训练模型时可以用：
# train: 2010-2018
# valid: 2019-2021
# test : 2022-2024
# ============================================================

panel["sample_split"] = "unused"

panel.loc[
    (panel["date"] >= "2010-01-01") & (panel["date"] <= "2018-12-31"),
    "sample_split"
] = "train"

panel.loc[
    (panel["date"] >= "2019-01-01") & (panel["date"] <= "2021-12-31"),
    "sample_split"
] = "valid"

panel.loc[
    (panel["date"] >= "2022-01-01") & (panel["date"] <= "2024-12-31"),
    "sample_split"
] = "test"


# ============================================================
# 9. 限制开发窗口，清理明显无效数据
# 这一步是在确保数据不包含 2025 之后的 held-out window。
# 另外，target_r_on_next 为空的行不能用于训练，但可以先保留在文件里。
# ============================================================

panel = panel[
    (panel["date"] >= START_DATE) &
    (panel["date"] <= END_DATE)
].copy()

panel = panel.sort_values(["date", "instrument_id"]).reset_index(drop=True)

# 可选：替换 inf 为 NaN，防止后面模型报错
panel = panel.replace([np.inf, -np.inf], np.nan)

print("\nPanel after date filtering:")
print(panel.shape)
print(panel["sample_split"].value_counts(dropna=False))


# ============================================================
# 10. 定义 Step 4 第一版模型建议使用的 feature list
# 这一步不是必须，但强烈建议保存下来。
# 后面写报告时可以清楚说明模型用了哪些变量。
# ============================================================

base_features = [
    # return / momentum / reversal features
    "r_on_today",
    "r_on_lag1",
    "r_id_lag1",
    "r_cc_lag1",
    "r_on_mean_5",
    "r_on_mean_20",
    "r_id_mean_5_lag1",
    "r_id_mean_20_lag1",
    "r_cc_mean_5_lag1",
    "r_cc_mean_20_lag1",
    "vol_20_lag1",
    "vol_60_lag1",
    "abs_r_on_today",
    "abs_r_cc_lag1",

    # Stage 2 liquidity / capacity features
    "price_for_filter",
    "adv20",
    "ann_vol60",
    "year_start_market_cap",
    "market_cap_rank",

    # short interest features
    "dsi",
    "dtcn",
    "ddtcn",

    # calendar features
    "month",
    "day_of_week",
    "is_month_end",
    "is_quarter_end",
]

borrow_features = [
    "hard_to_borrow_flag",
    "borrow_stress_flag",
    "borrow_rate_annual_bps",
    "borrow_cost_daily_bps",
]

# 如果 Stage 3 特征成功合并，就加入 feature list
if ADD_STAGE3_BORROW_FEATURES and borrow_path.exists():
    model_features = base_features + borrow_features
else:
    model_features = base_features

target_col = "target_r_on_next"

feature_config = {
    "target": target_col,
    "eligibility_filter": "is_trade_eligible == True",
    "date_window": {
        "start_date": START_DATE,
        "end_date": END_DATE,
    },
    "sample_split": {
        "train": "2010-01-01 to 2018-12-31",
        "valid": "2019-01-01 to 2021-12-31",
        "test": "2022-01-01 to 2024-12-31",
    },
    "features": model_features,
    "note": (
        "Features are constructed to avoid look-ahead bias. "
        "Close-based variables are shifted by one trading day. "
        "target_r_on_next is the next observed overnight return and is used only as label."
    )
}

with open(OUT_DIR / "stage4_feature_config.json", "w", encoding="utf-8") as f:
    json.dump(feature_config, f, indent=2)


# ============================================================
# 11. 保存最终 Step 4 建模数据文件
# 这一步会生成后面 Step 4 训练模型要用的主文件。
# ============================================================

output_path = OUT_DIR / "stage4_model_panel.parquet"
panel.to_parquet(output_path, index=False)

print("\nSaved Step 4 model panel to:")
print(output_path.resolve())


# ============================================================
# 12. 简单检查最终数据
# 这一步是在确认：
# - 有多少行
# - 有多少可交易行
# - target 覆盖率是多少
# - 每个 sample split 有多少可用样本
# ============================================================

eligible_panel = panel[panel["is_trade_eligible"] == True].copy()

summary = {
    "total_rows": len(panel),
    "eligible_rows": len(eligible_panel),
    "date_min": str(panel["date"].min().date()),
    "date_max": str(panel["date"].max().date()),
    "n_tickers": int(panel["ticker"].nunique()),
    "n_instruments": int(panel["instrument_id"].nunique()),
    "target_non_missing_rate_all": float(panel[target_col].notna().mean()),
    "target_non_missing_rate_eligible": float(eligible_panel[target_col].notna().mean()),
}

print("\nFinal panel summary:")
for k, v in summary.items():
    print(f"{k}: {v}")

print("\nEligible rows by sample split:")
print(
    eligible_panel
    .groupby("sample_split")[target_col]
    .agg(rows="size", target_non_missing=lambda x: x.notna().sum())
)

# 保存 summary，方便以后写报告或 debug
pd.Series(summary).to_json(OUT_DIR / "stage4_model_panel_summary.json", indent=2)

print("\nFeature config saved to:")
print((OUT_DIR / "stage4_feature_config.json").resolve())

In [2]:
from pathlib import Path
import os

# 当前 Jupyter kernel 的工作目录
cwd = Path.cwd()
print("Current working directory:")
print(cwd)

# 尝试自动定位 MLF_coursework 项目根目录
candidate_roots = [
    cwd,
    cwd / "MLF_coursework",
    *cwd.parents
]

base_dir = None

for root in candidate_roots:
    if (root / "panel_data_for_stage4").exists() or (root / "stage123data").exists() or (root / "data").exists():
        base_dir = root
        break

if base_dir is None:
    raise FileNotFoundError(
        "没有自动找到 MLF_coursework 项目根目录。请检查当前 notebook 是否在项目文件夹里运行。"
    )

print("\nDetected project root:")
print(base_dir)

# 搜索可能的 stage4 parquet 文件
parquet_candidates = list(base_dir.rglob("stage4_model_panel.parquet"))

# 如果没有精确文件名，就搜索所有 parquet，方便你看实际文件名
if len(parquet_candidates) == 0:
    print("\n没有找到 stage4_model_panel.parquet。下面列出项目里所有 parquet 文件：")
    all_parquets = list(base_dir.rglob("*.parquet"))
    for p in all_parquets:
        print(p.relative_to(base_dir))
    raise FileNotFoundError(
        "没有找到 stage4_model_panel.parquet。请看上面输出，确认你的 stage4 数据文件实际叫什么。"
    )

print("\nFound stage4_model_panel.parquet candidates:")
for p in parquet_candidates:
    print(p.relative_to(base_dir), "size:", round(p.stat().st_size / 1024**2, 2), "MB")

# 如果找到多个，默认选择最大的那个
panel_path = max(parquet_candidates, key=lambda p: p.stat().st_size)

print("\nUsing panel file:")
print(panel_path)

Current working directory:
c:\Users\yuchao\Desktop\MLF_coursework

Detected project root:
c:\Users\yuchao\Desktop\MLF_coursework

Found stage4_model_panel.parquet candidates:
panel_data_for_stage4\stage4_model_panel.parquet size: 833.94 MB

Using panel file:
c:\Users\yuchao\Desktop\MLF_coursework\panel_data_for_stage4\stage4_model_panel.parquet


所有特征

In [3]:
import pyarrow.parquet as pq
import pandas as pd

pf = pq.ParquetFile(panel_path)

print("Number of rows:", pf.metadata.num_rows)
print("Number of columns:", pf.metadata.num_columns)
print("Number of row groups:", pf.metadata.num_row_groups)

schema_df = pd.DataFrame({
    "column": pf.schema_arrow.names,
    "arrow_type": [str(field.type) for field in pf.schema_arrow]
})

display(schema_df)

# 保存所有列名
diagnostic_dir = panel_path.parent / "stage4_diagnostics"
diagnostic_dir.mkdir(exist_ok=True)

schema_df.to_csv(diagnostic_dir / "stage4_all_columns.csv", index=False)

print("Saved to:")
print(diagnostic_dir / "stage4_all_columns.csv")

Number of rows: 3773971
Number of columns: 55
Number of row groups: 4


,column,arrow_type
0,date,timestamp[ns]
1,year,int32
2,ticker,string
3,instrument_id,int64
4,in_universe,int64
5,market_cap,double
6,year_start_market_cap,double
7,market_cap_rank,double
8,price_for_filter,double
9,adv20,double


Saved to:
c:\Users\yuchao\Desktop\MLF_coursework\panel_data_for_stage4\stage4_diagnostics\stage4_all_columns.csv


用于step4的特征

In [4]:
import json

# 自动寻找 stage4_feature_config.json
config_candidates = list(base_dir.rglob("stage4_feature_config.json"))

config = {}
config_path = None

if len(config_candidates) > 0:
    # 优先选择和 panel 在同一个文件夹里的 config
    same_folder = [p for p in config_candidates if p.parent == panel_path.parent]
    
    if len(same_folder) > 0:
        config_path = same_folder[0]
    else:
        config_path = config_candidates[0]
    
    print("Using config file:")
    print(config_path)
    
    with open(config_path, "r", encoding="utf-8") as f:
        config = json.load(f)
else:
    print("没有找到 stage4_feature_config.json。下面只会输出所有列名，不能自动识别模型特征。")

# 兼容不同命名
target_col = config.get("target", config.get("target_column", None))

feature_cols = (
    config.get("features", None)
    or config.get("feature_columns", None)
    or config.get("model_features", None)
    or []
)

print("\nTarget column:")
print(target_col)

print("\nNumber of model features:", len(feature_cols))
print("\nModel features:")
for i, col in enumerate(feature_cols, 1):
    print(f"{i:02d}. {col}")

# 保存模型特征列表
with open(diagnostic_dir / "stage4_model_feature_list.txt", "w", encoding="utf-8") as f:
    f.write("Target column:\n")
    f.write(str(target_col) + "\n\n")
    f.write("Model features:\n")
    for i, col in enumerate(feature_cols, 1):
        f.write(f"{i:02d}. {col}\n")

print("\nSaved to:")
print(diagnostic_dir / "stage4_model_feature_list.txt")

Using config file:
c:\Users\yuchao\Desktop\MLF_coursework\panel_data_for_stage4\stage4_feature_config.json

Target column:
target_r_on_next

Number of model features: 26

Model features:
01. r_on_today
02. r_on_lag1
03. r_id_lag1
04. r_cc_lag1
05. r_on_mean_5
06. r_on_mean_20
07. r_id_mean_5_lag1
08. r_id_mean_20_lag1
09. r_cc_mean_5_lag1
10. r_cc_mean_20_lag1
11. vol_20_lag1
12. vol_60_lag1
13. abs_r_on_today
14. abs_r_cc_lag1
15. price_for_filter
16. adv20
17. ann_vol60
18. year_start_market_cap
19. market_cap_rank
20. dsi
21. dtcn
22. ddtcn
23. month
24. day_of_week
25. is_month_end
26. is_quarter_end

Saved to:
c:\Users\yuchao\Desktop\MLF_coursework\panel_data_for_stage4\stage4_diagnostics\stage4_model_feature_list.txt


可以用的特征：
FEATURE_COLS = [
    # liquidity / size / capacity proxies
    "year_start_market_cap",
    "market_cap_rank",
    "price_for_filter",
    "adv20",
    "ann_vol60",
    "impact_bps_at_participation_cap",

    # overnight / intraday / close-to-close lagged return features
    "r_on_today",
    "r_on_lag1",
    "r_id_lag1",
    "r_cc_lag1",
    "r_on_mean_5",
    "r_on_mean_20",
    "r_id_mean_5_lag1",
    "r_id_mean_20_lag1",
    "r_cc_mean_5_lag1",
    "r_cc_mean_20_lag1",
    "vol_20_lag1",
    "vol_60_lag1",
    "abs_r_on_today",
    "abs_r_cc_lag1",

    # short-interest / borrow stress proxies
    "dsi",
    "dtcn",
    "ddtcn",

    # calendar effects
    "month",
    "day_of_week",
    "is_month_end",
    "is_quarter_end",

不能用的特征：
DO_NOT_USE_AS_FEATURES = [
    "date",
    "year",
    "ticker",
    "instrument_id",
    "in_universe",
    "market_cap",
    "earnings_window",
    "eligibility_reason",
    "is_trade_eligible",
    "target_position_50m",
    "max_position_by_adv_50m",
    "position_limit_50m",
    "binding_constraint_50m",
    "target_position_250m",
    "max_position_by_adv_250m",
    "position_limit_250m",
    "binding_constraint_250m",
    "target_position_1000m",
    "max_position_by_adv_1000m",
    "position_limit_1000m",
    "binding_constraint_1000m",
    "open_adj",
    "close_adj",
    "r_on",
    "r_id",
    "r_cc",
    "target_r_on_next",
    "sample_split",
]

检查极端异常值和空值

In [5]:
import numpy as np
import pandas as pd
import gc

all_cols = pf.schema_arrow.names

rows = []
categorical_lines = []

for idx, col in enumerate(all_cols, 1):
    print(f"[{idx}/{len(all_cols)}] Checking column: {col}")
    
    s = pd.read_parquet(panel_path, columns=[col])[col]
    n = len(s)
    missing_n = int(s.isna().sum())
    missing_rate = missing_n / n if n > 0 else np.nan
    
    row = {
        "column": col,
        "pandas_dtype": str(s.dtype),
        "n_rows": n,
        "missing_n": missing_n,
        "missing_rate": missing_rate,
    }
    
    # 判断是否为数值列
    is_numeric = pd.api.types.is_numeric_dtype(s)
    is_bool = pd.api.types.is_bool_dtype(s)
    
    if is_numeric and not is_bool:
        x = pd.to_numeric(s, errors="coerce")
        x = x.replace([np.inf, -np.inf], np.nan)
        valid = x.dropna()
        
        row["valid_n"] = int(valid.shape[0])
        row["non_finite_n"] = int(pd.to_numeric(s, errors="coerce").isin([np.inf, -np.inf]).sum())
        
        if len(valid) > 0:
            q = valid.quantile([0.001, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999])
            
            row["min"] = valid.min()
            row["q0_1pct"] = q.loc[0.001]
            row["q1pct"] = q.loc[0.01]
            row["q5pct"] = q.loc[0.05]
            row["q25pct"] = q.loc[0.25]
            row["median"] = q.loc[0.5]
            row["q75pct"] = q.loc[0.75]
            row["q95pct"] = q.loc[0.95]
            row["q99pct"] = q.loc[0.99]
            row["q99_9pct"] = q.loc[0.999]
            row["max"] = valid.max()
            row["mean"] = valid.mean()
            row["std"] = valid.std()
            
            # 3倍IQR的极端异常值判断
            iqr = q.loc[0.75] - q.loc[0.25]
            lower = q.loc[0.25] - 3 * iqr
            upper = q.loc[0.75] + 3 * iqr
            
            row["iqr_extreme_lower"] = lower
            row["iqr_extreme_upper"] = upper
            row["iqr_extreme_n"] = int(((valid < lower) | (valid > upper)).sum())
            row["iqr_extreme_rate"] = row["iqr_extreme_n"] / len(valid)
            
            # 6倍标准差异常值
            if row["std"] is not None and row["std"] > 0:
                z6_n = int(((valid - row["mean"]).abs() > 6 * row["std"]).sum())
                row["z_abs_gt_6_n"] = z6_n
                row["z_abs_gt_6_rate"] = z6_n / len(valid)
            else:
                row["z_abs_gt_6_n"] = 0
                row["z_abs_gt_6_rate"] = 0
            
            # 对 return / target 类变量额外检查极端收益
            lower_col = col.lower()
            if ("return" in lower_col) or ("ret" in lower_col) or ("r_" in lower_col) or ("target" in lower_col):
                row["abs_gt_20pct_n"] = int((valid.abs() > 0.20).sum())
                row["abs_gt_50pct_n"] = int((valid.abs() > 0.50).sum())
                row["abs_gt_100pct_n"] = int((valid.abs() > 1.00).sum())
        else:
            row["valid_n"] = 0
    
    else:
        # 非数值列：记录前10个高频取值
        try:
            vc = s.astype("string").value_counts(dropna=False).head(10)
            categorical_lines.append(f"\n=== {col} ===")
            categorical_lines.append(str(vc))
        except Exception as e:
            categorical_lines.append(f"\n=== {col} ===")
            categorical_lines.append(f"Could not compute value counts: {e}")
    
    rows.append(row)
    
    del s
    gc.collect()

diagnostics = pd.DataFrame(rows)

# 标记列角色
def infer_role(col):
    if target_col is not None and col == target_col:
        return "target"
    elif col in feature_cols:
        return "model_feature"
    elif col in ["date", "year", "ticker", "instrument_id", "sample_split"]:
        return "id_or_split"
    elif "eligible" in col.lower() or "eligibility" in col.lower() or "reason" in col.lower():
        return "eligibility_filter"
    elif "borrow" in col.lower() or "tier" in col.lower():
        return "borrow_or_cost"
    elif "position" in col.lower() or "weight" in col.lower() or "cap" in col.lower():
        return "portfolio_construction"
    else:
        return "auxiliary"

diagnostics["role"] = diagnostics["column"].apply(infer_role)

# 保存结果
diagnostics.to_csv(diagnostic_dir / "stage4_missing_and_outlier_summary.csv", index=False)

with open(diagnostic_dir / "stage4_categorical_value_counts.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(categorical_lines))

print("\nSaved diagnostics to:")
print(diagnostic_dir / "stage4_missing_and_outlier_summary.csv")
print(diagnostic_dir / "stage4_categorical_value_counts.txt")

display(diagnostics)

[1/55] Checking column: date
[2/55] Checking column: year
[3/55] Checking column: ticker
[4/55] Checking column: instrument_id
[5/55] Checking column: in_universe
[6/55] Checking column: market_cap
[7/55] Checking column: year_start_market_cap
[8/55] Checking column: market_cap_rank
[9/55] Checking column: price_for_filter
[10/55] Checking column: adv20
[11/55] Checking column: ann_vol60
[12/55] Checking column: earnings_window
[13/55] Checking column: eligibility_reason
[14/55] Checking column: is_trade_eligible
[15/55] Checking column: impact_bps_at_participation_cap
[16/55] Checking column: target_position_50m
[17/55] Checking column: max_position_by_adv_50m
[18/55] Checking column: position_limit_50m
[19/55] Checking column: binding_constraint_50m
[20/55] Checking column: target_position_250m
[21/55] Checking column: max_position_by_adv_250m
[22/55] Checking column: position_limit_250m
[23/55] Checking column: binding_constraint_250m
[24/55] Checking column: target_position_1000m
[

,column,pandas_dtype,n_rows,missing_n,missing_rate,valid_n,non_finite_n,min,q0_1pct,q1pct,...,iqr_extreme_lower,iqr_extreme_upper,iqr_extreme_n,iqr_extreme_rate,z_abs_gt_6_n,z_abs_gt_6_rate,abs_gt_20pct_n,abs_gt_50pct_n,abs_gt_100pct_n,role
0,date,datetime64[ns],3773971,0,0.000000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,id_or_split
1,year,int32,3773971,0,0.000000,3773971.0,0.0,2.010000e+03,2.010000e+03,2.010000e+03,...,1.989000e+03,2.045000e+03,0.0,0.000000,0.0,0.000000,NaN,NaN,NaN,id_or_split
2,ticker,object,3773971,0,0.000000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,id_or_split
3,instrument_id,int64,3773971,0,0.000000,3773971.0,0.0,1.000000e+00,2.000000e+00,1.200000e+01,...,-2.093000e+03,3.430000e+03,0.0,0.000000,0.0,0.000000,NaN,NaN,NaN,id_or_split
4,in_universe,int64,3773971,0,0.000000,3773971.0,0.0,1.000000e+00,1.000000e+00,1.000000e+00,...,1.000000e+00,1.000000e+00,0.0,0.000000,0.0,0.000000,NaN,NaN,NaN,auxiliary
5,market_cap,float64,3773971,435,0.000115,3773536.0,0.0,1.181197e+08,2.939642e+08,5.161558e+08,...,-4.667295e+10,6.978188e+10,293686.0,0.077828,4141.0,0.001097,NaN,NaN,NaN,portfolio_construction
6,year_start_market_cap,float64,3773971,0,0.000000,3773971.0,0.0,2.564113e+08,2.881377e+08,4.997024e+08,...,-4.467886e+10,6.701797e+10,295854.0,0.078393,2262.0,0.000599,3773971.0,3773971.0,3773971.0,model_feature
7,market_cap_rank,float64,3773971,0,0.000000,3773971.0,0.0,1.000000e+00,1.970000e+00,1.070000e+01,...,-1.250000e+03,2.250000e+03,0.0,0.000000,0.0,0.000000,NaN,NaN,NaN,model_feature
8,price_for_filter,float64,3773971,1418,0.000376,3772553.0,0.0,2.036000e-01,1.375455e+00,4.280500e+00,...,-1.472050e+02,2.460095e+02,149491.0,0.039626,9798.0,0.002597,3772553.0,3771261.0,3769946.0,model_feature
9,adv20,float64,3773971,28360,0.007515,3745611.0,0.0,0.000000e+00,6.117767e+03,1.775110e+06,...,-4.409627e+08,6.463687e+08,238936.0,0.063791,9289.0,0.002480,NaN,NaN,NaN,model_feature
